<a href="https://colab.research.google.com/github/memo124/Laboratorio_IA_etica/blob/main/Laboratorio_IA_etica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Carga y Exploración de Datos

¡Hola! Bienvenidos a este cuaderno donde vamos a desentrañar los misterios del Titanic. En este primer paso, vamos a cargar nuestros datos y echarles un primer vistazo. Cada fila es una persona a bordo, y nuestro gran objetivo es predecir si esa persona `sobrevivió` (un `1`) o `no sobrevivió` (un `0`). ¡Manos a la obra!

In [ ]:
import pandas as pd
import numpy as np
import os

# Buscamos el archivo de forma local; si no está, lo descargamos automáticamente
file_path = 'train.csv'
if not os.path.exists(file_path):
    print("[INFO] 'train.csv' no se encontró localmente. Descargándolo desde la fuente pública...")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.to_csv(file_path, index=False)
else:
    print("[INFO] Cargando 'train.csv' desde el almacenamiento local...")
    df = pd.read_csv(file_path)

# Una mirada rápida para entender qué pinta tienen los datos
print("=== VISTA PREVIA DE LOS PASAJEROS ===")
display(df.head())

# Revisamos tipos de datos y la presencia de valores nulos
print("\n=== INFORMACIÓN GENERAL DEL DATASET ===")
df.info()

print("\n=== CANTIDAD DE VALORES FALTANTES POR COLUMNA ===")
print(df.isna().sum())

## Paso 2: División Estratégica del Dataset

Para ser justos con nuestro modelo y evitar sorpresas desagradables, vamos a dividir nuestros datos del Titanic en tres grupos importantes, ¡como si estuviéramos repartiendo cartas!

1.  **Entrenamiento (60%)**: Aquí es donde nuestro modelo va a 'estudiar' y aprender de los patrones. Piensa que es su fase de aprendizaje.
2.  **Validación (20%)**: Este es nuestro 'campo de pruebas'. Lo usaremos para afinar el modelo, probar diferentes configuraciones y ver qué funciona mejor sin tocar el set final.
3.  **Prueba (20%)**: ¡El examen final! Este conjunto lo guardamos bajo llave y solo lo usaremos una vez, al final, para saber qué tan bien le iría a nuestro modelo con datos totalmente nuevos. Así nos aseguramos de que no hay 'trampas' y la evaluación es lo más realista posible.

In [ ]:
from sklearn.model_selection import train_test_split

# 'Survived' es la etiqueta que queremos predecir; el resto son características
X = df.drop(columns=['Survived'])
y = df['Survived']

# Primero aislamos el 20% para el test final.
# Usamos stratify para asegurarnos de que la proporción de sobrevivientes se mantenga.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Del 80% restante, tomamos el 25% para validación (que equivale al 20% del total original)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print("=== REPARTO DE LOS CONJUNTOS DE DATOS ===")
print(f"Entrenamiento (X_train): {X_train.shape} | ({round(len(X_train)/len(df)*100)}% del total)")
print(f"Validación    (X_val):   {X_val.shape} | ({round(len(X_val)/len(df)*100)}% del total)")
print(f"Prueba        (X_test):  {X_test.shape} | ({round(len(X_test)/len(df)*100)}% del total)")

## Paso 3: Preprocesamiento de Datos Básicos

¡Es hora de limpiar y preparar nuestros datos! Imagina que estamos poniendo a punto un motor. Aquí vamos a construir un 'pipeline' (una tubería de pasos) para manejar los números y las categorías. La clave es que todas las 'reglas' de limpieza (como rellenar datos faltantes con la mediana o lo más común) las aprenderemos **solo** de nuestro set de entrenamiento. ¡No queremos 'espiar' los datos de validación o prueba! Así, cuando nuestro modelo se enfrente a datos nuevos, estará listo para lo que sea.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Columnas seleccionadas para iniciar nuestro modelo
selected_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train_filtered = X_train[selected_cols]
X_val_filtered = X_val[selected_cols]
X_test_filtered = X_test[selected_cols]

# Separamos variables por su naturaleza para aplicar diferentes transformaciones
num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Pclass', 'Sex', 'Embarked']

# Pipeline numérico: Imputamos valores ausentes con la mediana
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Pipeline categórico: Imputamos con el más común y aplicamos codificación OneHot
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Integramos ambos procesos en un único transformador de columnas
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# ¡MUY IMPORTANTE! Ajustamos (fit) solo con entrenamiento para no contaminar el proceso
X_train_prepared = preprocessor.fit_transform(X_train_filtered)
X_val_prepared = preprocessor.transform(X_val_filtered)
X_test_prepared = preprocessor.transform(X_test_filtered)

# Recuperamos el nombre de las columnas generadas para mantener la claridad
cat_features = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)
feature_names = num_cols + list(cat_features)

print("=== PROCESO DE PREPARACIÓN COMPLETADO ===")
print(f"Total de columnas resultantes: {len(feature_names)}")
print(f"Nombres de las columnas: {feature_names}")

## Paso 4: Entrenamiento y Evaluación del Modelo Baseline

¡Llegó el momento de que nuestro primer modelo entre en acción! Vamos a entrenar un `RandomForestClassifier` (piensa en él como un 'bosque de decisiones' que toma nuestro ordenador). Este será nuestro 'modelo base' o 'baseline'. Es como el punto de partida. Evaluaremos su rendimiento en los datos de validación usando algo llamado **F1-Score**. Este número nos dirá qué tan bien lo hizo este primer intento, y será nuestra referencia para saber si las mejoras futuras realmente valen la pena. ¡A ver qué tal le va!

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# Creamos el modelo de referencia con una semilla para reproducibilidad
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train_prepared, y_train)

# Realizamos predicciones sobre el conjunto de validación que apartamos previamente
y_val_pred = baseline_model.predict(X_val_prepared)

# Medimos el rendimiento del modelo baseline
val_f1 = f1_score(y_val, y_val_pred)

print("=== RESULTADOS DEL MODELO BASELINE ===")
print(f"F1-Score en Validación: {val_f1:.4f}")
print("\nReporte Detallado de Métricas de Clasificación:")
print(classification_report(y_val, y_val_pred))